# Phase 2 grading training (Kaggle, GPU)

Trains the DR severity grading model (CORN ordinal head over a timm backbone) on the
**full** APTOS 2019 dataset. Per `AGENTS.md`, heavy grading-model training only happens
here, not locally on the Apple M4 dev machine -- the local repo only trains the small
IQA CNN and MA patch classifier.

**Setup before running:**
1. In the notebook's right sidebar, add a dataset: attach an APTOS 2019 Kaggle dataset
   (e.g. `mariaherrerot/aptos2019`, or the official `aptos2019-blindness-detection`
   competition if you've accepted its rules). Note the slug you attach.
2. Turn on a GPU accelerator (Settings -> Accelerator -> GPU T4 x2 or P100).
3. Turn on internet access (Settings -> Internet -> On) so this can `git clone`.

**Session death / restart:** training checkpoints every epoch to
`models/grading/epoch_N.pt` under `/kaggle/working`, and `src.drscreen.grading.train`
automatically resumes from the latest checkpoint. If a session dies mid-run, just
re-run this notebook top to bottom -- the cache-build and clone steps are idempotent,
and training will pick up where it left off as long as `/kaggle/working` survived the
restart (true for a kernel restart within the same session; a brand-new session starts
with an empty `/kaggle/working` and trains from scratch).

In [ ]:
# Kaggle's base image doesn't ship every package this project needs.
!pip install -q timm albumentations onnx onnxscript

In [ ]:
import os

REPO_URL = "https://github.com/prithvipm412/SugarEyes.git"
REPO_DIR = "/kaggle/working/SugarEyes"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    # idempotent re-run after a session restart: bring the code up to date,
    # but never touch models/, runs/, or data/ -- that's where our resumable
    # training state and cache live.
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

In [ ]:
# Point the repo's data/raw/aptos/ at whatever APTOS dataset is attached under
# /kaggle/input -- reusing the same keyword-aware CSV/image-dir matching the
# local download script uses (this mirror's train_1.csv/valid.csv/test.csv +
# nested train_images/train_images/ layout tripped up a naive glob once already;
# see scripts/download_aptos_sample.py's _find_label_csv / _find_image_dir).
import shutil
from pathlib import Path

import pandas as pd

from scripts.download_aptos_sample import _find_label_csv, _find_image_dir

KAGGLE_INPUT = Path("/kaggle/input")
input_dirs = [p for p in KAGGLE_INPUT.iterdir() if p.is_dir()]
assert input_dirs, "No dataset attached under /kaggle/input -- attach an APTOS 2019 dataset in the sidebar first."
dataset_dir = input_dirs[0]
print(f"using attached dataset: {dataset_dir}")

label_csv = _find_label_csv(dataset_dir)
image_dir = _find_image_dir(dataset_dir, keyword="train")
print(f"label csv: {label_csv}")
print(f"image dir: {image_dir}")

labels = pd.read_csv(label_csv)
id_col = "id_code" if "id_code" in labels.columns else labels.columns[0]
grade_col = "diagnosis" if "diagnosis" in labels.columns else labels.columns[1]
labels = labels[[id_col, grade_col]].rename(columns={id_col: "id", grade_col: "grade"})

raw_dir = Path("data/raw/aptos")
images_out = raw_dir / "images"
images_out.mkdir(parents=True, exist_ok=True)

rows = []
for _, row in labels.iterrows():
    matches = list(image_dir.glob(f"{row['id']}.*"))
    if not matches:
        continue
    dst = images_out / matches[0].name
    if not dst.exists():
        # symlink, not copy -- Kaggle input is read-only and this avoids doubling disk usage
        os.symlink(matches[0], dst)
    rows.append({"filename": matches[0].name, "grade": int(row["grade"])})

assert rows, "No images matched between the label CSV and image directory -- check the dataset layout."
pd.DataFrame(rows).to_csv(raw_dir / "labels.csv", index=False)
print(f"wrote {len(rows)} full-dataset images + labels to {raw_dir}")

In [ ]:
# Full-dataset cache (no --sample) -- writes data/cache/manifest.csv with every image.
!python -m scripts.build_cache --datasets aptos

In [ ]:
# Also write the committed reproducibility split (data/splits/aptos_{train,val}.csv).
# Uses the same seed=42 stratified split as build_cache's own internal split, so the
# two agree -- this just produces the artifact meant to be committed back to the repo.
!python -m scripts.make_splits

In [ ]:
# Trains for configs/grading_kaggle.yaml's epoch count, checkpointing every epoch to
# models/grading/epoch_N.pt and models/grading/best.pt. Auto-resumes if checkpoints
# already exist (re-run this cell after any interruption).
!python -m src.drscreen.grading.train --config configs/grading_kaggle.yaml

In [ ]:
# Export the best checkpoint to ONNX (opset 13) for local CPU/MPS deployment.
import torch
import yaml

from src.drscreen.grading.model import GradingModel, export_onnx

with open("configs/grading_kaggle.yaml") as f:
    config = yaml.safe_load(f)

model = GradingModel(backbone_name=config["backbone"], pretrained=False)
checkpoint = torch.load("models/grading/best.pt", map_location="cpu")
model.load_state_dict(checkpoint["model_state"])
print(f"exporting checkpoint from epoch {checkpoint['epoch']}, val_qwk={checkpoint['val_qwk']:.4f}")

export_onnx(model, "models/grading.onnx", image_size=config.get("image_size", 224))
print("wrote models/grading.onnx")

## After this notebook finishes

`models/grading/best.pt`, `models/grading.onnx`, and `data/splits/aptos_train.csv` /
`aptos_val.csv` are now in `/kaggle/working`. None of these get committed to git
automatically (checkpoints/ONNX are gitignored by design -- see `AGENTS.md`).

1. Download `models/grading/best.pt` and `models/grading.onnx` from this notebook's
   Output tab and place them at the same paths in the local repo.
2. Commit `data/splits/aptos_train.csv` / `aptos_val.csv` back to the repo -- those
   *are* meant to be committed, as the reproducibility artifact for this exact split.
3. Locally, run `python -m scripts.select_threshold --config configs/grading_kaggle.yaml`
   to freeze the referable-DR threshold on validation, then
   `python -m scripts.check_parity` to verify ONNX/PyTorch agreement, then
   `python -m tests.gate_phase2`.
4. Only once Messidor-2 is available locally (see `data/splits/README.md` -- it needs
   ADCIS registration): run `python -m scripts.evaluate` exactly once for the real
   headline sensitivity/specificity numbers.